# QPINN: stationary Hamilton-Jacobi

Domain: $\Omega=(-1,1)^2$ with homogeneous Dirichlet boundary condition $u=0$ on $\partial\Omega$.

PDE:

$$
-\Delta u(x)+\rho\|\nabla u(x)\|_2^\beta-q(x)=\alpha_0,
\qquad x=(x_1,x_2)\in\Omega.
$$

Source:

$$
q(x_1,x_2)=\sum_{(m,n)} c_{mn}
\cos\left(\frac{m\pi x_1}{2}\right)
\cos\left(\frac{n\pi x_2}{2}\right),
$$

with modes $(1,1),(1,3),(3,1),(1,5),(5,1)$ and coefficients defined below. This PDE has no closed-form analytical solution in the notebook; plots use a finite-difference Newton-Krylov reference solution.

Models: Ordinary QFM, Cheb-QFM, exact $B_2$-QCM, randomized $B_2$-QCM, and a fully connected PINN baseline. The shared circuit, PDE loss, training, and plotting code lives in the `qmps` package.


## Parameters

Set the PDE, circuit, FC PINN, training, output-directory, and figure-resolution parameters.


In [28]:
from __future__ import annotations

import qmps as qmp
import qmps.experiments as qexp
from qmps.problems import StationaryHamiltonJacobiProblem

# Runtime.
USE_DOUBLE_PRECISION = False
SEED = 5

# Problem and circuit size.
DIM = 2
N_QUBITS = 2
N_UPLOAD_LAYERS = 2
STRONG_LAYERS_PER_BLOCK = 3

# Circuit encoding and output scaling.
EXP_BASE = 3.0
ENCODING_SCALE = 1.0
OUTPUT_SCALE = 0.75
DIFF_GENERATOR_PER_LAYER = False
ACOS_EPS = 1.0e-6

# Fully connected PINN baseline.
FC_HIDDEN_LAYERS = (4, 7)

# Stationary viscous Hamilton-Jacobi PDE.
ALPHA0 = 0.0
HJ_RHO = 0.15
HJ_GRAD_POWER = 1.5
HJ_GRAD_EPS = 1.0e-12
HJ_SOURCE_MODE_COEFFICIENTS = {
    (1, 1): 0.50,
    (1, 3): 0.15,
    (3, 1): 0.15,
    (1, 5): 0.20,
    (5, 1): 0.20,
}
REFERENCE_NEWTON_TOL = 1.0e-10

# Randomized B2 average.
RANDOMIZED_B2_SAMPLES = 6

# Training.
TRAINABLE = True
TRAINING_STEPS = 300
N_RUNS = 1
INTERIOR_BATCH = 50
BOUNDARY_BATCH = 50
LR = 1.0e-1
LR_DECAY = 0.99
LR_MIN = 1.0e-4
INTERIOR_MARGIN = 0.96
GRAD_CLIP = 5.0
SOFT_BOUNDARY_WEIGHT = 10.0

# Evaluation and plotting.
EVAL_GRID_N = 50
FIGURE_DIR = "figures"
TRAINING_DATA_DIR = "training_data/hj"
FIGURE_DPI = 300  # Use 600 for high-resolution export.
LOSS_YLIM = (1.0e-3, 1.6e2)
LOSS_YTICKS = [1.0e-3, 1.0e-2, 1.0e-1, 1.0, 1.0e1, 1.0e2]
FIGURE_PREFIX = "qpinn_hj"

runtime = qexp.setup_runtime(
    USE_DOUBLE_PRECISION,
    SEED,
    figure_dir=FIGURE_DIR,
    data_dir=TRAINING_DATA_DIR,
)
problem = StationaryHamiltonJacobiProblem(
    alpha0=ALPHA0,
    rho=HJ_RHO,
    grad_power=HJ_GRAD_POWER,
    grad_eps=HJ_GRAD_EPS,
    reference_newton_tol=REFERENCE_NEWTON_TOL,
    source_mode_coefficients=HJ_SOURCE_MODE_COEFFICIENTS,
)
qfm_config = qmp.QFMConfig(
    dim=DIM,
    n_qubits=N_QUBITS,
    n_upload_layers=N_UPLOAD_LAYERS,
    strong_layers_per_block=STRONG_LAYERS_PER_BLOCK,
    exp_base=EXP_BASE,
    encoding_scale=ENCODING_SCALE,
    output_scale=OUTPUT_SCALE,
    diff_generator_per_layer=DIFF_GENERATOR_PER_LAYER,
    acos_eps=ACOS_EPS,
    randomized_b2_samples=RANDOMIZED_B2_SAMPLES,
)
fc_config = qexp.FullyConnectedPINNConfig(
    dim=DIM,
    hidden_layers=FC_HIDDEN_LAYERS,
)
training_config = qexp.TrainingConfig(
    steps=TRAINING_STEPS,
    n_runs=N_RUNS,
    interior_batch=INTERIOR_BATCH,
    boundary_batch=BOUNDARY_BATCH,
    lr=LR,
    lr_decay=LR_DECAY,
    lr_min=LR_MIN,
    interior_margin=INTERIOR_MARGIN,
    grad_clip=GRAD_CLIP,
)
MODEL_SPECS = {
    "Ordinary QFM": {"group": "none", "input_map": "raw"},
    "Cheb-QFM": {"group": "none", "input_map": "chebyshev"},
    "$B_2$-QCM": {"group": "hyperoctahedral", "input_map": "chebyshev"},
    "Randomized $B_2$-QCM": {
        "group": "randomized_hyperoctahedral",
        "input_map": "chebyshev",
        "group_sample_count": RANDOMIZED_B2_SAMPLES,
    },
}


Using device=mps, real dtype=torch.float32, state representation=real/imag pair


## Models And Losses

Build the four quantum model variants, the FC PINN baseline, and the hard/soft loss functions.


In [29]:
def make_models(seed: int = SEED, hard_boundary: bool = True):
    return qmp.make_qfm_models(
        qfm_config,
        MODEL_SPECS,
        seed=seed,
        hard_boundary=hard_boundary,
        dtype=runtime.dtype,
        device=runtime.device,
    )


def make_fc_models(seed: int = SEED, hard_boundary: bool = True):
    return qexp.make_fully_connected_pinn_models(
        fc_config,
        seed=seed,
        hard_boundary=hard_boundary,
        dtype=runtime.dtype,
        device=runtime.device,
    )


def hard_loss(model, interior_x, boundary_x):
    return problem.loss(model, interior_x, boundary_x, boundary_weight=0.0)


def soft_loss(model, interior_x, boundary_x):
    return problem.loss(model, interior_x, boundary_x, boundary_weight=SOFT_BOUNDARY_WEIGHT)


loss_functions = qexp.make_loss_functions(hard_loss, soft_loss)
models = make_models(seed=SEED, hard_boundary=True)
fc_models = make_fc_models(seed=SEED, hard_boundary=True)
qexp.print_parameter_counts({**models, **fc_models})


Trainable parameter counts:
  Ordinary QFM            : 54
  Cheb-QFM                : 54
  $B_2$-QCM               : 54
  Randomized $B_2$-QCM    : 54
  Fully connected PINN    : 55


## Training Data

Train only when `TRAINABLE = True`; otherwise load the saved local single-run training-data files for plotting.


In [ ]:
if TRAINABLE:
    result = qexp.train_and_save_boundary_comparison_runs(
        make_models=make_models,
        make_fc_models=make_fc_models,
        loss_functions=loss_functions,
        model_names=list(MODEL_SPECS),
        config=training_config,
        runtime=runtime,
        dim=DIM,
        seed=SEED,
        problem=problem,
        grid_n=EVAL_GRID_N,
        figure_prefix=FIGURE_PREFIX,
    )
else:
    result = qexp.load_training_data(
        runtime.data_dir,
        FIGURE_PREFIX,
        make_models=make_models,
        make_fc_models=make_fc_models,
    )
    training_data_paths = qexp.available_training_data_files(runtime.data_dir, FIGURE_PREFIX)
    for training_data_path in training_data_paths:
        print(f"loaded training data: {training_data_path}")



=== Run 5/5 | init_seed=4005, batch_seed=4142 ===
Hard constraint  | Ordinary QFM             run  5/5, step    1/300: lr=1.000e-01, loss=6.458e+00, residual=6.458e+00, boundary=1.602e-16
Hard constraint  | Ordinary QFM             run  5/5, step   60/300: lr=5.527e-02, loss=3.290e-02, residual=3.290e-02, boundary=1.314e-17
Hard constraint  | Ordinary QFM             run  5/5, step  120/300: lr=3.024e-02, loss=1.122e-02, residual=1.122e-02, boundary=9.864e-18
Hard constraint  | Ordinary QFM             run  5/5, step  180/300: lr=1.655e-02, loss=9.586e-03, residual=9.586e-03, boundary=6.790e-18
Hard constraint  | Ordinary QFM             run  5/5, step  240/300: lr=9.053e-03, loss=1.301e-02, residual=1.301e-02, boundary=8.575e-18


## Figures

Save the side-by-side loss figure and the combined solution-comparison figure. The solution panels compare the reference solution with the representative trained model for each constraint/model; the representative run is selected by final training residual for hard constraint and final training objective for soft constraint. With `N_RUNS = 1`, every panel uses run 1.


In [ ]:
figure_paths = [
    qexp.plot_final_loss_comparison(
        result,
        runtime.figure_dir,
        FIGURE_PREFIX,
        dpi=FIGURE_DPI,
        loss_ylim=LOSS_YLIM,
        loss_yticks=LOSS_YTICKS,
    ),
    qexp.plot_final_solution_comparison(
        result,
        problem,
        runtime,
        dim=DIM,
        grid_n=EVAL_GRID_N,
        figure_prefix=FIGURE_PREFIX,
        dpi=FIGURE_DPI,
    ),
]
for figure_path in figure_paths:
    print(f"saved figure: {figure_path}")

qexp.print_final_metric_summary(result)
